# CLIP Fine-Tune — WikiArt Style Classification

**Runtime**: GPU → Runtime > Change runtime type > T4 GPU  
**Saves**: checkpoint + metrics to Google Drive at `MyDrive/clip_wikiart/`

| Step | Cell |
|------|------|
| Mount Drive | 1 |
| Install dependencies | 2 |
| Optional HF token input | 3 |
| Download WikiArt from HF | 4 |
| Build dataset + dataloader | 5 |
| Fine-tune CLIP ViT-B/32 | 6 |
| Save checkpoint + metrics | 7 |

In [ ]:
# ── Cell 1: Mount Drive + create output dir ──────────────────────────────────
import os

DRIVE_MOUNTED = False
OUT_DIR = '/content/clip_wikiart'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    OUT_DIR = '/content/drive/MyDrive/clip_wikiart'
    DRIVE_MOUNTED = True
    print('Google Drive mounted successfully.')
except Exception as exc:
    print(f'[warn] Drive mount failed: {exc}')
    print('[warn] Falling back to local runtime storage at /content/clip_wikiart')
    print('[warn] You can still train, but download artifacts before the runtime ends.')

os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output dir: /content/drive/MyDrive/clip_wikiart


In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# Colab already has torch/torchvision; just need transformers + datasets
%pip install -q transformers datasets pyarrow pillow tqdm

: 

In [ ]:
# ── Cell 3: Optional secure HF token input (runtime only) ─────────────────────
from getpass import getpass
from huggingface_hub import login, whoami
import os

# Leave blank and press Enter to run unauthenticated.
token_input = getpass('HF_TOKEN (optional, hidden input): ').strip()
if token_input:
    os.environ['HF_TOKEN'] = token_input
    os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
    HUB_TOKEN = token_input
    login(token=token_input, add_to_git_credential=False, skip_if_logged_in=False)
    try:
        user = whoami(token=HUB_TOKEN).get('name', 'unknown')
        print(f'HF token set for this runtime session. Authenticated as: {user}')
    except Exception as exc:
        print(f'HF token set, but identity check failed: {exc}')
else:
    HUB_TOKEN = False
    os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
    print('No token entered. Continuing unauthenticated.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF token set for this runtime session. Authenticated as: sgromk


In [ ]:
# ── Cell 4: Download WikiArt from HuggingFace ─────────────────────────────────
# Streams directly from pre-converted Parquet shards — no full download, no OOM.
# Note: the HF parquet mirror for this dataset is partial, so we adapt split sizes
# to the number of rows actually available instead of assuming the full ~81k rows.
from datasets import load_dataset
from huggingface_hub import list_repo_files
import json
import os
import urllib.parse
import urllib.request

# Prevent implicit Colab Secrets lookup (causes vault timeout warnings in VS Code).
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'

repo_id = 'huggan/wikiart'
REQUESTED_TRAIN = 38000
REQUESTED_VAL   = 4750
SEED            = 42

# Priority: runtime input token -> env var -> unauthenticated
runtime_token = globals().get('HUB_TOKEN', False)
if isinstance(runtime_token, str) and runtime_token.strip():
    hf_token = runtime_token.strip()
else:
    hf_token = os.getenv('HF_TOKEN', '').strip()

token_arg = hf_token if hf_token else False
HUB_TOKEN = token_arg  # shared by later cells
print('Using authenticated Hub requests.' if hf_token else 'Continuing unauthenticated.')

# Enumerate Parquet shards from refs/convert/parquet (lazy — no image data downloaded)
print('Listing Parquet shards...')
all_files = list_repo_files(
    repo_id=repo_id,
    repo_type='dataset',
    revision='refs/convert/parquet',
    token=HUB_TOKEN,
)
parquet_files = sorted(
    f"hf://datasets/{repo_id}@refs/convert/parquet/{p}"
    for p in all_files
    if p.endswith('.parquet')
)
if not parquet_files:
    raise RuntimeError('No parquet shards found at refs/convert/parquet — check repo access.')

print(f'Found {len(parquet_files)} shards. Building streaming dataset (no data downloaded yet)...')
raw = load_dataset(
    'parquet',
    data_files={'train': parquet_files},
    split='train',
    streaming=True,
    token=HUB_TOKEN,
)
print('Dataset ready (parquet streaming — images fetched lazily in Cell 5)')

# Query dataset-server metadata so split sizes match the rows actually available.
size_url = f"https://datasets-server.huggingface.co/size?dataset={urllib.parse.quote(repo_id, safe='')}"
available_rows = None
is_partial = False
try:
    with urllib.request.urlopen(size_url, timeout=30) as resp:
        size_info = json.load(resp)
    dataset_size = size_info.get('size', {}).get('dataset', {})
    available_rows = dataset_size.get('num_rows')
    is_partial = bool(size_info.get('partial'))
except Exception as exc:
    print(f'[warn] Could not fetch dataset size metadata: {exc}')

if isinstance(available_rows, int) and available_rows > 0:
    if available_rows >= REQUESTED_TRAIN + REQUESTED_VAL:
        MAX_TRAIN = REQUESTED_TRAIN
        MAX_VAL = REQUESTED_VAL
    else:
        # Keep roughly a 90/10 split when the parquet mirror is partial.
        MAX_TRAIN = max(1, int(available_rows * 0.9))
        MAX_VAL = max(1, available_rows - MAX_TRAIN)
        print(
            f'[note] Parquet mirror is partial ({available_rows:,} rows available; '
            f'requested {REQUESTED_TRAIN + REQUESTED_VAL:,}). '
            f'Using MAX_TRAIN={MAX_TRAIN:,}, MAX_VAL={MAX_VAL:,}.'
        )
else:
    MAX_TRAIN = REQUESTED_TRAIN
    MAX_VAL = REQUESTED_VAL

print(f'Config -> train: {MAX_TRAIN:,}  val: {MAX_VAL:,}  partial_mirror: {is_partial}')


In [ ]:
# ── Cell 5: Dataset class + dataloader ───────────────────────────────────────
import os
import warnings
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor
from PIL import Image
from io import BytesIO
import random
from pathlib import Path

# Ensure no implicit Colab Secrets lookup in this cell
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
local_token = globals().get('HUB_TOKEN', False)

# Build auth kwargs for current transformers versions
auth_kwargs = {}
if isinstance(local_token, str) and local_token:
    auth_kwargs['token'] = local_token
else:
    auth_kwargs['token'] = False

# ---- label map (alphabetical = integer id, matches huggan/wikiart) ----------
STYLE_NAMES = [
    'Abstract Expressionism', 'Action Painting', 'Analytical Cubism',
    'Art Nouveau', 'Baroque', 'Color Field Painting', 'Contemporary Realism',
    'Cubism', 'Early Renaissance', 'Expressionism', 'Fauvism',
    'High Renaissance', 'Impressionism', 'Mannerism Late Renaissance',
    'Minimalism', 'Naive Art Primitivism', 'New Realism',
    'Northern Renaissance', 'Pointillism', 'Pop Art', 'Post Impressionism',
    'Realism', 'Rococo', 'Romanticism', 'Symbolism', 'Synthetic Cubism', 'Ukiyo-e'
]
N_CLASSES = len(STYLE_NAMES)  # 27

MODEL_ID  = 'openai/clip-vit-base-patch32'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  N_CLASSES: {N_CLASSES}')

# Suppress only the noisy Colab-vault warning emitted by huggingface_hub
with warnings.catch_warnings():
    warnings.filterwarnings(
        'ignore',
        message=r'.*HF_TOKEN.*secret value from your vault.*',
    )
    warnings.filterwarnings(
        'ignore',
        message=r'.*You are not authenticated with the Hugging Face Hub.*',
    )
    processor = CLIPProcessor.from_pretrained(MODEL_ID, use_fast=False, **auth_kwargs)

CLIP_SIZE = (224, 224)  # CLIP input resolution — resize here to save RAM

def decode_and_shrink(image_field, size=CLIP_SIZE, quality=85):
    """Decode any HF image field, resize to CLIP input size, re-encode as compact JPEG."""
    if isinstance(image_field, dict):
        raw_bytes = image_field.get('bytes')
        path      = image_field.get('path')
        if raw_bytes is not None:
            img = Image.open(BytesIO(raw_bytes)).convert('RGB')
        elif path is not None:
            img = Image.open(path).convert('RGB')
        else:
            raise TypeError(f'image dict has neither bytes nor path: {list(image_field.keys())}')
    elif isinstance(image_field, (bytes, bytearray)):
        img = Image.open(BytesIO(image_field)).convert('RGB')
    elif isinstance(image_field, Image.Image):
        img = image_field.convert('RGB')
    else:
        raise TypeError(f'Unsupported image field type: {type(image_field)}')
    img = img.resize(size, Image.LANCZOS)
    buf = BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()

def bytes_to_pil(image_bytes):
    return Image.open(BytesIO(image_bytes)).convert('RGB')

# ---- cache buffered rows to avoid re-buffering on reruns --------------------
cache_dir = Path(OUT_DIR) / 'cache'
cache_dir.mkdir(parents=True, exist_ok=True)
cache_file = cache_dir / f'wikiart_rows_train{MAX_TRAIN}_val{MAX_VAL}_seed{SEED}_224q85.pt'

all_rows = None
if cache_file.exists():
    print(f'Loading cached buffered rows from: {cache_file}')
    all_rows = torch.load(cache_file, map_location='cpu')
    print(f'Loaded {len(all_rows):,} rows from cache')

if all_rows is None:
    print('Buffering train+val rows (images resized to 224×224 to save RAM)...')
    random.seed(SEED)
    target_rows = MAX_TRAIN + MAX_VAL

    all_rows = []
    bad_rows = 0
    for i, row in enumerate(raw):
        try:
            image_bytes = decode_and_shrink(row['image'])
            all_rows.append({'image_bytes': image_bytes, 'style': int(row['style'])})
        except Exception as exc:
            bad_rows += 1
            if bad_rows <= 5:
                print(f'[warn] skipping row {i}: {type(row.get("image", None))} | {exc}')
            continue

        if len(all_rows) % 5000 == 0:
            print(f'  buffered {len(all_rows):,}/{target_rows:,} rows...')

        if len(all_rows) >= target_rows:
            break

    print(f'Buffered {len(all_rows):,} usable rows (skipped {bad_rows:,})')
    torch.save(all_rows, cache_file)
    print(f'Saved buffer cache to: {cache_file}')

random.seed(SEED)
random.shuffle(all_rows)

actual_total = len(all_rows)
requested_total = MAX_TRAIN + MAX_VAL
if actual_total < requested_total:
    if actual_total < 2:
        raise RuntimeError(f'Only {actual_total} rows available after buffering; need at least 2 to split train/val.')
    old_train, old_val = MAX_TRAIN, MAX_VAL
    MAX_TRAIN = max(1, int(actual_total * 0.9))
    MAX_VAL = max(1, actual_total - MAX_TRAIN)
    print(
        f'[note] Buffered fewer rows than requested ({actual_total:,} < {requested_total:,}). '
        f'Adjusting split from train/val={old_train:,}/{old_val:,} to {MAX_TRAIN:,}/{MAX_VAL:,}.'
    )

train_rows = all_rows[:MAX_TRAIN]
val_rows   = all_rows[MAX_TRAIN:MAX_TRAIN + MAX_VAL]
print(f'Train: {len(train_rows):,}  Val: {len(val_rows):,}')

class WikiArtStyleDataset(Dataset):
    def __init__(self, rows, processor):
        self.rows = rows
        self.processor = processor

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        image = bytes_to_pil(row['image_bytes'])
        label = row['style']
        image_inputs = self.processor(
            images=image,
            return_tensors='pt',
        )
        return {
            'pixel_values': image_inputs['pixel_values'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long),
        }

# ---- fast-mode dataloader with non-fatal teardown suppression ----------------
# NOTE: >0 workers can trigger a noisy, non-fatal teardown AssertionError in
# Colab/Jupyter: "can only test a child process". We suppress only that case.
from torch.utils.data import dataloader as _dl
_orig_del = getattr(_dl._MultiProcessingDataLoaderIter, '__del__', None)

def _quiet_del(self):
    try:
        if _orig_del is not None:
            _orig_del(self)
    except AssertionError as exc:
        if 'can only test a child process' in str(exc):
            return
        raise

_dl._MultiProcessingDataLoaderIter.__del__ = _quiet_del

BATCH_SIZE = 32
NUM_WORKERS = 4
PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2

train_ds = WikiArtStyleDataset(train_rows, processor)
val_ds = WikiArtStyleDataset(val_rows, processor)

if NUM_WORKERS > 0:
    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        prefetch_factor=PREFETCH_FACTOR,
    )
    val_dl = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        prefetch_factor=PREFETCH_FACTOR,
    )
else:
    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=PIN_MEMORY,
    )
    val_dl = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=PIN_MEMORY,
    )

print(
    f'Train batches: {len(train_dl)}  Val batches: {len(val_dl)}  '
    f'num_workers: {NUM_WORKERS}  pin_memory: {PIN_MEMORY}  '
    f'persistent_workers: {PERSISTENT_WORKERS}'
)

In [ ]:
# ── Cell 6: Fine-tune CLIP ────────────────────────────────────────────────────
from transformers import CLIPModel
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import json

# ---- Hyperparameters --------------------------------------------------------
EPOCHS        = 5
LR            = 1e-5
WARMUP_STEPS  = 200
UNFREEZE_AFTER = 1   # freeze visual encoder for epoch 0, unfreeze after

# ---- Model ------------------------------------------------------------------
local_token = globals().get('HUB_TOKEN', False)
model_auth_kwargs = {}
if isinstance(local_token, str) and local_token:
    model_auth_kwargs['token'] = local_token
else:
    model_auth_kwargs['token'] = False

clip = CLIPModel.from_pretrained(MODEL_ID, **model_auth_kwargs).to(DEVICE)

# Add a linear classification head on top of image features
embed_dim = clip.config.projection_dim  # 512 for ViT-B/32
classifier = nn.Linear(embed_dim, N_CLASSES).to(DEVICE)

def get_image_embeddings(clip_model, pixel_values):
    """Return projected image embeddings as Tensor (B, projection_dim)."""
    vision_outputs = clip_model.vision_model(pixel_values=pixel_values)
    pooled = vision_outputs.pooler_output
    return clip_model.visual_projection(pooled)

# Freeze visual encoder initially
for p in clip.vision_model.parameters():
    p.requires_grad = False

optimizer = AdamW(
    list(clip.visual_projection.parameters()) + list(classifier.parameters()),
    lr=LR, weight_decay=0.01
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_dl))
criterion = nn.CrossEntropyLoss()

history = []

for epoch in range(EPOCHS):
    # Unfreeze visual encoder after first epoch
    if epoch == UNFREEZE_AFTER:
        print('Unfreezing vision encoder...')
        for p in clip.vision_model.parameters():
            p.requires_grad = True
        optimizer.add_param_group({'params': clip.vision_model.parameters(), 'lr': LR * 0.1})

    # ---- Train
    clip.train(); classifier.train()
    train_loss = 0.0; train_correct = 0; train_total = 0

    for batch in tqdm(train_dl, desc=f'Epoch {epoch+1}/{EPOCHS} [train]'):
        pv  = batch['pixel_values'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        img_feats = get_image_embeddings(clip, pv)  # (B, projection_dim)
        logits = classifier(img_feats)              # (B, 27)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_loss    += loss.item() * labels.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    # ---- Val
    clip.eval(); classifier.eval()
    val_loss = 0.0; val_correct = 0; val_total = 0

    with torch.no_grad():
        for batch in tqdm(val_dl, desc=f'Epoch {epoch+1}/{EPOCHS} [val]'):
            pv     = batch['pixel_values'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            img_feats = get_image_embeddings(clip, pv)
            logits    = classifier(img_feats)
            loss      = criterion(logits, labels)
            val_loss    += loss.item() * labels.size(0)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)

    row = {
        'epoch':     epoch + 1,
        'train_loss': round(train_loss / train_total, 4),
        'train_acc':  round(train_correct / train_total, 4),
        'val_loss':   round(val_loss / val_total, 4),
        'val_acc':    round(val_correct / val_total, 4),
    }
    history.append(row)
    print(f"Epoch {epoch+1}: train_loss={row['train_loss']}  train_acc={row['train_acc']}  "
          f"val_loss={row['val_loss']}  val_acc={row['val_acc']}")

In [ ]:
# ── Cell 7: Save checkpoint + metrics to Drive ───────────────────────────────
import json, os
from pathlib import Path

if not history:
    raise RuntimeError('No training history found. Run Cell 6 (training) successfully before saving.')

ckpt_dir = Path(OUT_DIR)
ckpt_dir.mkdir(parents=True, exist_ok=True)

# Save CLIP model + classifier head
clip.save_pretrained(str(ckpt_dir / 'clip_finetuned'))
processor.save_pretrained(str(ckpt_dir / 'clip_finetuned'))
torch.save(classifier.state_dict(), str(ckpt_dir / 'classifier_head.pt'))

# Save training history + config
config = {
    'base_model': MODEL_ID,
    'n_classes': N_CLASSES,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'train_size': MAX_TRAIN,
    'val_size': MAX_VAL,
    'style_names': STYLE_NAMES,
    'history': history,
    'best_val_acc': max(r['val_acc'] for r in history),
    'best_epoch': max(history, key=lambda r: r['val_acc'])['epoch'],
}
(ckpt_dir / 'training_config.json').write_text(json.dumps(config, indent=2))

print(f'Checkpoint saved to {ckpt_dir}')
print(f"Best val accuracy: {config['best_val_acc']:.4f} (epoch {config['best_epoch']})")

# ---- Quick sanity check: reload and run one forward pass -------------------
from transformers import CLIPModel as _CLIPModel

loaded_clip = _CLIPModel.from_pretrained(str(ckpt_dir / 'clip_finetuned')).to(DEVICE)
loaded_head = nn.Linear(embed_dim, N_CLASSES).to(DEVICE)
loaded_head.load_state_dict(torch.load(str(ckpt_dir / 'classifier_head.pt'), map_location=DEVICE))
loaded_clip.eval(); loaded_head.eval()

sample_batch = next(iter(val_dl))
with torch.no_grad():
    vision_outputs = loaded_clip.vision_model(pixel_values=sample_batch['pixel_values'].to(DEVICE))
    feats = loaded_clip.visual_projection(vision_outputs.pooler_output)
    preds = loaded_head(feats).argmax(1).cpu().tolist()
true_labels = sample_batch['label'].tolist()
acc = sum(p == t for p, t in zip(preds, true_labels)) / len(preds)
print(f'Sanity-check batch accuracy: {acc:.3f}  (expected ~{config["best_val_acc"]:.3f})')
print('Done!')

## To download the checkpoint back to your local machine

After running on Colab, your files are at `MyDrive/clip_wikiart/`. You can:

1. **Sync via Google Drive desktop** — fastest, no extra steps if Drive is mounted locally
2. **Download manually** from [drive.google.com](https://drive.google.com) → `clip_wikiart/`
3. **From a local terminal** (if you have `rclone` configured):
   ```bash
   rclone copy gdrive:clip_wikiart artifacts/checkpoints/clip_wikiart
   ```

The checkpoint folder contains:
- `clip_finetuned/` — full HuggingFace model (loadable with `CLIPModel.from_pretrained`)
- `classifier_head.pt` — linear head weights (27-class)
- `training_config.json` — hyperparams + per-epoch metrics